# Mini Batch Gradient Descent

In [1]:
from sklearn.datasets import load_diabetes

In [2]:
x, y = load_diabetes(return_X_y=True)

In [3]:
print(x.shape)
print(y.shape)

(442, 10)
(442,)


### Train test split

In [4]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=2)

print(x_train.shape)
print(y_train.shape)

(353, 10)
(353,)


### Base Model

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

reg = LinearRegression()

reg.fit(x_train, y_train)

print("Intercept: ", reg.intercept_)
print("Coefficients: ", reg.coef_)

print("R2 Score: ", r2_score(y_test, reg.predict(x_test)))

Intercept:  151.88331005254167
Coefficients:  [  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238]
R2 Score:  0.439933866156897


### Custom Mini batch class

In [7]:
import numpy as np
import random

In [8]:
class MBGD:
    def __init__(self, batch_size = 1, learning_rate=0.01, epochs=100):
        self.intercept_ = None
        self.coef_ = None
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

    def fit(self, x_train, y_train):
        self.intercept_ = 0
        self.coef_ = np.ones(x_train.shape[1])

        for i in range(self.epochs):
            for j in range(int(x_train.shape[0] / self.batch_size)):
                idx = random.sample(range(x_train.shape[0]), self.batch_size)

                y_hat = np.dot(x_train[idx], self.coef_) + self.intercept_
                intercept_der = -2 * np.mean(y_train[idx] - y_hat)
                self.intercept_ = self.intercept_ - (self.lr * intercept_der)

                coef_der = -2 * np.dot((y_train[idx] - y_hat), x_train[idx])
                self.coef_ = self.coef_ - (self.lr * coef_der)

        print("Intercept: ", self.intercept_)
        print("Coefficients: ", self.coef_)

    def predict(self, x_test):
        return np.dot(x_test, self.coef_) + self.intercept_

In [18]:
mbgd = MBGD(batch_size=int(x_train.shape[0] / 50), learning_rate=0.01, epochs=100)

In [19]:
mbgd.fit(x_train, y_train)

print("R2 Score: ", r2_score(y_test, mbgd.predict(x_test)))

Intercept:  152.41293911735022
Coefficients:  [  39.09577918 -143.76207526  455.10340692  309.13020011  -22.61818923
  -92.36701164 -188.17933886  110.47025591  409.82511449  107.90848985]
R2 Score:  0.45237314284872987


### Mini batch GD Regressor in sklearn

In [43]:
from sklearn.linear_model import SGDRegressor

sgd = SGDRegressor(learning_rate="constant", eta0=0.25)

batch_size = 35
epochs = 100
for i in range(epochs):
    idx = random.sample(range(x_train.shape[0]), batch_size)
    sgd.partial_fit(x_train[idx], y_train[idx])

In [44]:
print("Intercept: ", sgd.intercept_)
print("Coefficients: ", sgd.coef_)

print("R2 Score: ", r2_score(y_test, sgd.predict(x_test)))

Intercept:  [142.23012632]
Coefficients:  [  15.71246753 -130.10441883  466.20961387  319.63657296  -46.12851211
 -116.46952225 -192.55600492  104.50587575  419.47056286   81.59960776]
R2 Score:  0.43133650369904153
